# Bluebikes GBFS Snapshot

Pulls the latest `station_information` and `station_status` feeds from the
[Bluebikes GBFS](https://gbfs.bluebikes.com/gbfs/gbfs.json) and caches each as
`data/raw/{feed_name}/{pulled_at}/data.csv`.

GBFS is live data, so every run is a new snapshot: pulls are keyed by UTC pull
timestamp rather than loaded from cache.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from shared.api import get

## Params

In [ ]:
GBFS_URL = "https://gbfs.bluebikes.com/gbfs/gbfs.json"
FEEDS = ["station_information", "station_status"]
LANGUAGE = "en"

## Pull and cache

Feed URLs are discovered from the `gbfs.json` auto-discovery file rather than
hardcoded. Nested fields (e.g. `rental_uris`) are flattened into dotted
columns.

In [ ]:
feed_urls = {
    feed["name"]: feed["url"]
    for feed in get(GBFS_URL)["data"][LANGUAGE]["feeds"]
}

pulled_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")

tables = {}
for feed_name in FEEDS:
    payload = get(feed_urls[feed_name])
    df = pd.json_normalize(payload["data"]["stations"])
    df["last_updated"] = pd.to_datetime(payload["last_updated"], unit="s", utc=True)

    raw_path = Path(f"data/raw/{feed_name}/{pulled_at}/data.csv")
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(raw_path, index=False)
    print(f"Fetched and cached {len(df):,} rows: {raw_path}")

    tables[feed_name] = df

station_information = tables["station_information"]
station_status = tables["station_status"]
station_information.head()

In [ ]:
station_status.head()